[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Estadística descriptiva, forma de la distribución y valores atípicos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Miércoles 21 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo II: *Describir y comparar*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se describirán distribuciones con medidas clásicas y robustas (media, mediana, desviación estándar, IQR y MAD), se analizará su forma (asimetría, curtosis y multimodalidad) con histogramas, KDE, ECDF y diagramas de caja, y se detectarán valores atípicos con la regla del IQR y el puntaje *z* robusto, usando `penguins_limpio`.

## 🎯 Objetivo de la sesión.

**Describir distribuciones con medidas robustas y detectar valores atípicos sin eliminarlos por reflejo.**

Al terminar podrás:
1. Calcular e interpretar media, mediana, desviación estándar, IQR y MAD, y saber cuáles son robustas.
2. Describir la forma de una distribución (asimetría, curtosis, multimodalidad) y elegir la gráfica adecuada: histograma, KDE, ECDF o diagrama de caja.
3. Detectar valores atípicos con la regla del IQR, el puntaje *z* y el puntaje *z* robusto, y decidir si son errores o valores reales.
4. Aplicar una transformación logarítmica y explicar su efecto.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Repaso: `penguins_limpio` |
| 25 | 1. Tendencia central y dispersión |
| 25 | 2. Forma de la distribución |
| 25 | 3. Valores atípicos |
| 15 | 4. Transformaciones (log) |
| 20 | 5. Ejercicio integrador y cierre |

🟢 = ejercicio básico · 🔴 = reto opcional

## 📚 Referencia de la API: curso py311.

Esta sesión usa métodos ya vistos en py311 sin repasarlos uno por uno: `quantile`, `value_counts` y `nlargest` en [py311 · 19 Herramientas de análisis de datos](https://github.com/PythonistaMX/py311/blob/main/19_herramientas_de_analisis_de_datos.ipynb); `transform` y `apply` en [py311 · 17 Los métodos `apply()` y `transform()`](https://github.com/PythonistaMX/py311/blob/main/17_metodos_apply_y_transform.ipynb); `groupby` en [py311 · 24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb); y las gráficas de distribución de *Seaborn* en [py311 · 31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb). El tiempo de la sesión se dedica a **decidir** qué medida describe mejor cada distribución y qué hacer con los atípicos.

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3, así que pueden verse diferencias (por ejemplo, texto como `object` en lugar de `str`, o el uso de `inplace=True`). Cuando difieran, se sigue lo que muestra este *notebook* (ver la sección 2.1 del *notebook* 01).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluido el módulo `stats` de *SciPy*, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats

warnings.filterwarnings("ignore", message="vert: bool was deprecated")

**Nota:** el filtro de advertencias oculta solo un aviso interno de *Seaborn* 0.13 con *Matplotlib* 3.11 al dibujar diagramas de caja (`vert: bool was deprecated`). No afecta los resultados.

---
## 0. Repaso: `penguins_limpio` (10 min).

En la sesión 2 se generó `data/penguins_limpio.csv`: `sex` homologado a `Male`/`Female`, sin las 2 filas que no tenían medidas y con `sex` faltante sin imputar. Si el archivo no existe (por ejemplo, porque no se ejecutó el *notebook* 02), se regenera con las mismas decisiones.

* La siguiente celda leerá `penguins_limpio.csv` o lo regenerará desde `penguins.csv` si no existe.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio).astype({"species": "category", "island": "category", "sex": "category"})
pinguinos.shape

* La siguiente celda mostrará los tipos y los faltantes que quedan.

In [ ]:
pd.DataFrame({"dtype": pinguinos.dtypes.astype(str), "faltantes": pinguinos.isna().sum()})

---
## 1. Tendencia central y dispersión (25 min).

| Medida | Qué resume | ¿Robusta? | En código |
|---|---|---|---|
| Media | Centro de gravedad | No: un solo valor extremo la arrastra | `s.mean()` |
| Mediana | Valor que deja la mitad de los datos a cada lado | Sí | `s.median()` |
| Desviación estándar | Distancia típica a la media | No | `s.std()` (con `ddof=1`) |
| IQR (rango intercuartílico) | Ancho del 50% central: Q3 − Q1 | Sí | `stats.iqr(s)` |
| MAD (desviación absoluta mediana) | Mediana de \|x − mediana\| | Sí | `stats.median_abs_deviation(s, scale="normal")` |

Una medida es **robusta** si no cambia mucho cuando una pequeña parte de los datos es extrema o errónea. Con `scale="normal"`, la MAD se multiplica por 1.4826 para que sea comparable con la desviación estándar cuando los datos son normales.

**Concepto: fórmulas de centro y dispersión.** Con datos ordenados $x_{(1)} \leq \dots \leq x_{(n)}$:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad IQR = Q_3 - Q_1 \qquad MAD = 1.4826 \cdot \operatorname{mediana}\left(|x_i - \tilde{x}|\right)$$

donde $\tilde{x}$ es la mediana. La media y $s$ usan **todos los valores con su magnitud**, por eso un valor extremo las arrastra; la mediana, el IQR y la MAD solo dependen del **orden** de los valores centrales. En el flujo del EDA, describir el centro y la dispersión es el primer resumen de cada variable clave, y la forma (sección 2) decide cuál de las dos parejas reportar.

* La siguiente celda extraerá la masa de los Adelie.

In [ ]:
masa_adelie = pinguinos.loc[pinguinos["species"] == "Adelie", "body_mass_g"]
masa_adelie.describe()

* La siguiente celda calculará las cinco medidas para la masa de los Adelie.

In [ ]:
pd.Series({
    "media": masa_adelie.mean(),
    "mediana": masa_adelie.median(),
    "desv_est": masa_adelie.std(),
    "iqr": stats.iqr(masa_adelie),
    "mad": stats.median_abs_deviation(masa_adelie, scale="normal"),
}).round(1)

**Interpretación:** media (3,700.7 g) y mediana (3,700 g) casi coinciden, y la desviación estándar (458.6 g) y la MAD (444.8 g) también: es la firma de una distribución **casi simétrica y sin atípicos**. En ese caso ambas parejas dicen lo mismo y se puede reportar media y desviación estándar. El IQR (650 g) es mayor que $s$ porque mide el ancho del 50% central, no una "distancia típica"; para una normal, $IQR \approx 1.35\,s$.

**Ejemplo:** se simulará un error de captura: alguien escribe 37,500 g en lugar de 3,750 g. Se compararán las medidas con y sin ese único valor.

* La siguiente celda agregará el valor erróneo y comparará las medidas.

In [ ]:
con_error = pd.concat([masa_adelie, pd.Series([37500])], ignore_index=True)

def medidas_basicas(serie):
    return pd.Series({
        "media": serie.mean(),
        "mediana": serie.median(),
        "desv_est": serie.std(),
        "iqr": stats.iqr(serie),
        "mad": stats.median_abs_deviation(serie, scale="normal"),
    })

pd.DataFrame({"original": medidas_basicas(masa_adelie), "con un error": medidas_basicas(con_error)}).round(1)

Un solo valor erróneo entre 152 sube la media en más de 200 g (de 3,701 a 3,923 g) y multiplica por seis la desviación estándar (de 459 a 2,779 g), mientras que la mediana, el IQR y la MAD no se mueven. Por eso, ante datos sospechosos, conviene reportar **mediana e IQR** junto con media y desviación estándar.

### 🟢 Ejercicio 1 · Una función de resumen.

**🎯 Objetivo:** escribir una función que reúna medidas sensibles y robustas. **Qué se espera aprender:** a reportar siempre centro y dispersión en pareja, y a reutilizar el código en lugar de copiarlo.

Escribe `resumen(serie)` que devuelva una `Series` con `n` (no nulos), `media`, `mediana`, `desv_est`, `iqr` y `mad` (con `scale="normal"`), ignorando los faltantes. Aplícala a `body_mass_g` y a `flipper_length_mm` de todos los pingüinos y muestra ambos resultados en un *dataframe*.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ
def resumen(serie):
    return None

### 🟢 Ejercicio 2 · Descripción por especie.

**🎯 Objetivo:** aplicar el resumen dentro de cada grupo. **Qué se espera aprender:** que una descripción global puede ocultar grupos con centros y dispersiones muy distintos.

Describe `body_mass_g` y `flipper_length_mm` **por especie** con las mismas medidas (pista: `groupby("species")[columna].apply(resumen).unstack()`, o `agg` con una lista de funciones). Compara con el resumen global del ejercicio 1: ¿qué medida global describe mal a **todas** las especies?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Punto de quiebre.

**🎯 Objetivo:** medir cuánta contaminación tolera cada medida. **Qué se espera aprender:** el concepto de punto de quiebre, que explica por qué la mediana es robusta y la media no.

Toma la masa de los Gentoo y reemplaza una proporción creciente de valores (0%, 5%, 10%, …, 45%) por un valor absurdo (60,000 g). Para cada proporción calcula media, mediana, desviación estándar y MAD, y grafica cómo cambian. ¿A partir de qué proporción deja de servir la mediana? (El **punto de quiebre** de la media es 0%; el de la mediana, 50%.)

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 1:**
* Media y desviación estándar usan la magnitud de todos los valores; mediana, IQR y MAD solo el orden: son **robustas**.
* Un solo error puede mover mucho la media y la desviación estándar.
* Se reporta la pareja adecuada a la forma, o ambas con justificación.

---
## 2. Forma de la distribución (25 min).

* **Asimetría** (*skewness*): positiva si la cola larga está a la derecha (media > mediana), negativa si está a la izquierda. Cerca de 0 indica simetría.
* **Curtosis** (en exceso): compara el peso de las colas con una normal (0). Positiva = colas más pesadas, más valores extremos.
* **Multimodalidad:** más de un pico. Suele indicar que se mezclan **grupos distintos**.

`pandas` calcula ambas con `s.skew()` y `s.kurt()` (versiones corregidas por sesgo); *SciPy* ofrece `stats.skew` y `stats.kurtosis`.

* La siguiente celda calculará la asimetría y la curtosis de las cuatro medidas.

In [ ]:
pd.DataFrame({"asimetria": pinguinos[medidas].skew(), "curtosis": pinguinos[medidas].kurt()}).round(2)

**Interpretación:** las cuatro medidas tienen asimetría pequeña (−0.14 a 0.47) y curtosis negativa (−0.72 a −0.98): colas más ligeras que una normal. Una curtosis negativa tan marcada en las cuatro es sospechosa: suele indicar una **mezcla de grupos** con picos separados, que "aplana" la distribución. Las gráficas lo confirmarán.

**Fórmulas:** la asimetría es $g_1 = \frac{1}{n}\sum (x_i - \bar{x})^3 / s^3$ y la curtosis en exceso, $g_2 = \frac{1}{n}\sum (x_i - \bar{x})^4 / s^4 - 3$ (*pandas* aplica una corrección por sesgo). Elevar al cubo conserva el signo de las desviaciones; elevar a la cuarta da peso a las colas.

### 2.1 Cuatro gráficas para una variable.

| Gráfica | Fortaleza | Cuidado |
|---|---|---|
| Histograma | Intuitivo; muestra picos y huecos | Depende del número de *bins* |
| KDE (densidad suavizada) | Curva continua; fácil de superponer por grupo | Depende del ancho de banda; puede "inventar" valores fuera del rango |
| ECDF (distribución acumulada empírica) | Sin parámetros; se leen percentiles directamente | Menos intuitiva al principio |
| Diagrama de caja | Mediana, IQR y atípicos de un vistazo; compara muchos grupos | Oculta la multimodalidad |

* La siguiente celda dibujará las cuatro gráficas de `flipper_length_mm` para todos los pingüinos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(pinguinos, x="flipper_length_mm", bins=30, ax=axes[0, 0]).set(title="Histograma")
sns.kdeplot(pinguinos, x="flipper_length_mm", fill=True, ax=axes[0, 1]).set(title="KDE")
sns.ecdfplot(pinguinos, x="flipper_length_mm", ax=axes[1, 0]).set(title="ECDF")
sns.boxplot(pinguinos, x="flipper_length_mm", ax=axes[1, 1]).set(title="Diagrama de caja")
plt.tight_layout()
plt.show()

**Interpretación:** el histograma y la KDE muestran **dos picos** (alrededor de 190 mm y de 210 a 215 mm), la ECDF tiene un tramo casi plano entre ellos y el diagrama de caja **no muestra nada raro**: una caja simétrica sin atípicos. Es el ejemplo de por qué no basta con una gráfica: el diagrama de caja oculta la bimodalidad.

El histograma y la KDE muestran **dos picos**; el diagrama de caja los esconde por completo. Por eso nunca conviene describir una variable solo con un diagrama de caja.

### 🟢 Ejercicio 3 · ¿Por qué la distribución global es bimodal?

**🎯 Objetivo:** explicar una forma de la distribución con una variable de grupo. **Qué se espera aprender:** que la multimodalidad suele indicar una mezcla de poblaciones.

Grafica la distribución de `flipper_length_mm` y de `body_mass_g` **separada por especie** (pista: `sns.histplot(..., hue="species")` o `sns.kdeplot(..., hue="species")`). Explica por qué la distribución global se ve bimodal y qué cambia al separar por especie.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 4 · Forma por especie.

**🎯 Objetivo:** comparar la forma de la distribución entre grupos. **Qué se espera aprender:** a calcular asimetría y curtosis por grupo y a decidir qué medidas reportar.

1. Calcula la asimetría y la curtosis de `body_mass_g` **por especie** y compara la media con la mediana en cada una.
2. Dibuja la ECDF de `body_mass_g` por especie. Con la gráfica, estima qué proporción de Gentoo pesa menos de 5,000 g y compruébalo con código.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 2 · ¿Cuántos picos? Depende de cómo se mire.

**🎯 Objetivo:** ver cómo los parámetros de una gráfica cambian lo que parece mostrar. **Qué se espera aprender:** a no sacar conclusiones de un solo número de *bins* o ancho de banda.

Dibuja la KDE global de `flipper_length_mm` con `bw_adjust` igual a 0.2, 1 y 3, y el histograma con 5, 20 y 60 *bins*. ¿Cuándo aparecen o desaparecen picos? ¿Qué recomendarías para no engañarte?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 2:**
* Asimetría (dirección de la cola), curtosis (peso de las colas) y número de picos describen la forma.
* Ninguna gráfica basta sola: el diagrama de caja oculta la multimodalidad; el histograma depende de los *bins*.
* La bimodalidad suele indicar grupos mezclados.

---
## 3. Valores atípicos (25 min).

Un valor atípico (*outlier*) es una observación muy alejada del resto. **No es sinónimo de error**: puede ser un error de captura, un caso real extremo o la señal de que se mezclan grupos.

| Regla | Atípico si… | Ventaja | Problema |
|---|---|---|---|
| IQR (Tukey) | x < Q1 − 1.5·IQR o x > Q3 + 1.5·IQR | Robusta; es la del diagrama de caja | Marca muchos en distribuciones asimétricas |
| Puntaje *z* | \|x − media\| / desv_est > 3 | Simple | La media y la desviación estándar se contaminan con los propios atípicos (*enmascaramiento*) |
| *z* robusto | \|x − mediana\| / MAD > 3.5 | Robusta | Supone una distribución aproximadamente simétrica |

El umbral de 3.5 para el *z* robusto es el que recomiendan Iglewicz y Hoaglin (1993).

**Concepto:** las tres reglas comparan la distancia de cada valor al centro con una medida de dispersión:

$$z_i = \frac{x_i - \bar{x}}{s} \qquad z^{\text{robusto}}_i = \frac{x_i - \tilde{x}}{MAD} \qquad \text{IQR: } x_i \notin [Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$$

La diferencia clave es **qué centro y qué dispersión** usan: si son sensibles a los atípicos, los propios atípicos agrandan la vara de medir y dejan de parecer extremos.

* La siguiente celda definirá las tres reglas como funciones que devuelven una máscara booleana.

In [ ]:
def atipicos_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    return (serie < q1 - k * (q3 - q1)) | (serie > q3 + k * (q3 - q1))

def atipicos_z(serie, umbral=3):
    return ((serie - serie.mean()) / serie.std()).abs() > umbral

def z_robusto(serie):
    return (serie - serie.median()) / stats.median_abs_deviation(serie, scale="normal", nan_policy="omit")

def atipicos_z_robusto(serie, umbral=3.5):
    return z_robusto(serie).abs() > umbral

**Ejemplo:** se simularán 200 mediciones normales (media 200, desviación 10) más 25 errores de captura cercanos a 300, para ver qué detecta cada regla.

* La siguiente celda creará la muestra contaminada y contará cuántos atípicos marca cada regla.

In [ ]:
simulada = pd.Series(np.concatenate([rng.normal(200, 10, size=200), rng.normal(300, 3, size=25)]))
pd.Series({
    "IQR": atipicos_iqr(simulada).sum(),
    "z > 3": atipicos_z(simulada).sum(),
    "z robusto > 3.5": atipicos_z_robusto(simulada).sum(),
})

Las reglas robustas encuentran los 25 errores (la del IQR puede marcar, además, algún valor legítimo extremo); el puntaje *z* clásico no encuentra **ninguno**. Es el efecto de **enmascaramiento**.

* La siguiente celda mostrará por qué falla el puntaje *z*: los errores inflan la media y la desviación estándar.

In [ ]:
pd.DataFrame({"limpia": medidas_basicas(simulada[:200]), "contaminada": medidas_basicas(simulada)}).round(1)

**Interpretación:** los 25 errores (11% de los datos) suben la desviación estándar de 9.3 a 33.1. Un error en 300 queda a unas $(300 - 210) / 33.1 \approx 2.7$ desviaciones de la media contaminada: por debajo de 3, así que no se marca. La mediana y la MAD apenas cambian, y por eso el *z* robusto sí los detecta.

### ¿Error o valor real?

Antes de tocar un atípico conviene preguntarse:
1. ¿Es **imposible** (masa negativa, aleta de 2 m)? → error: corregir si la causa es clara o pasar a `NaN` (sesión 2).
2. ¿Es **plausible** pero extremo? → se conserva y se reporta; se puede analizar con y sin él.
3. ¿Es atípico **solo en el conjunto global** y normal dentro de su grupo? → no es atípico: se están mezclando grupos.
4. ¿Hay forma de **verificar** con la fuente? → esa es la mejor opción.

**Advertencia:** borrar atípicos "por reflejo" para que los datos se vean normales es una forma de manipulación. Toda exclusión se documenta y se justifica.

### 🟢 Ejercicio 5 · Atípicos en `penguins`.

**🎯 Objetivo:** aplicar las reglas de atípicos globalmente y por especie. **Qué se espera aprender:** que un valor puede ser atípico dentro de su grupo y normal en el conjunto, y viceversa.

Para `body_mass_g` y `flipper_length_mm`:
1. Marca los atípicos con la regla del IQR y con el *z* robusto, **globalmente** y **dentro de cada especie** (pista: `pinguinos.groupby("species")[columna].transform(atipicos_iqr)`).
2. Compara las listas: ¿qué filas marca cada regla? ¿Coinciden?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda mostrará las filas marcadas por al menos una regla (requiere `marcas`).

In [ ]:
if "marcas" in globals():
    marcadas = marcas.any(axis=1)
    columnas_mostrar = ["species", "sex", "body_mass_g", "flipper_length_mm"]
    resultado = pinguinos.loc[marcadas, columnas_mostrar].join(marcas.loc[marcadas].astype(int).T.loc[lambda d: d.any(axis=1)].T)
else:
    resultado = None
resultado

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 6 · ¿Error o valor real?

**🎯 Objetivo:** decidir qué hacer con los atípicos encontrados. **Qué se espera aprender:** a justificar con conocimiento del dominio si se corrige, se elimina o se conserva un valor.

Con base en la tabla anterior y en lo que sabes de los pingüinos, decide para cada atípico encontrado si lo tratarías como error o como valor real, y qué harías con él en el análisis.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · Atípicos que solo existen dentro del grupo.

**🎯 Objetivo:** buscar atípicos condicionados al grupo. **Qué se espera aprender:** a detectar valores raros que una revisión global no ve.

Encuentra los pingüinos cuyo `bill_length_mm` es atípico **dentro de su especie** según el *z* robusto, pero **no** lo es en el conjunto global (o al revés). Grafícalos resaltados sobre un diagrama de dispersión `bill_length_mm` vs. `bill_depth_mm` coloreado por especie. ¿Por qué ocurre?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 3:**
* Atípico no es sinónimo de error: se investiga antes de decidir.
* El puntaje *z* clásico sufre enmascaramiento; el *z* robusto y el IQR resisten la contaminación.
* Conviene buscar atípicos también **dentro** de cada grupo.

---
## 4. Transformaciones (15 min).

Muchas variables positivas (ingresos, precios, tiempos, tamaños de población) tienen **asimetría positiva** fuerte: la mayoría de los valores son pequeños y unos pocos son enormes. El **logaritmo** comprime la cola derecha:

* Convierte diferencias multiplicativas en aditivas (×10 pasa a ser +1 en `log10`).
* Reduce la asimetría y la influencia de los valores grandes.
* Solo se aplica a valores positivos; con ceros se usa `np.log1p(x)` = log(1 + x).
* Al volver a la escala original, la media de los logaritmos corresponde a la **media geométrica**, no a la media aritmética.

`penguins` es casi simétrico, así que se simularán ingresos mensuales con una distribución lognormal.

* La siguiente celda simulará 1,000 ingresos mensuales (en pesos) y su logaritmo base 10.

In [ ]:
ingresos = pd.Series(rng.lognormal(mean=np.log(15000), sigma=0.8, size=1000), name="ingreso")
log_ingresos = np.log10(ingresos)
pd.DataFrame({
    "ingreso": [ingresos.skew(), ingresos.mean(), ingresos.median(), atipicos_iqr(ingresos).sum()],
    "log10(ingreso)": [log_ingresos.skew(), log_ingresos.mean(), log_ingresos.median(), atipicos_iqr(log_ingresos).sum()],
}, index=["asimetria", "media", "mediana", "atipicos IQR"]).round(2)

**Interpretación:** en pesos, la asimetría es 2.31, la media (20,066) supera en un tercio a la mediana (14,919) y la regla del IQR marca 63 "atípicos". En $\log_{10}$, la asimetría es prácticamente 0, media y mediana coinciden y quedan solo 4 marcas. Los datos son los mismos; la escala cambia lo que parece extremo.

* La siguiente celda comparará ambas distribuciones gráficamente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(ingresos, bins=40, ax=axes[0]).set(title="Ingreso (escala original)", xlabel="pesos")
sns.histplot(log_ingresos, bins=40, ax=axes[1]).set(title="log10(ingreso)", xlabel="log10(pesos)")
plt.tight_layout()
plt.show()

* La siguiente celda comparará la media aritmética con la media geométrica (10 elevado a la media de los logaritmos).

In [ ]:
{"media aritmética": round(ingresos.mean()), "media geométrica": round(10 ** log_ingresos.mean()), "mediana": round(ingresos.median())}

**Interpretación:** la media geométrica ($10^{\overline{\log_{10} x}} \approx$ 14,602 pesos) queda junto a la mediana, mientras que la aritmética (20,066) la arrastran los ingresos altos. Para variables multiplicativas y asimétricas, la media geométrica o la mediana describen mejor el "ingreso típico".

En la escala original, la regla del IQR marca decenas de "atípicos" que son simplemente la cola natural de la distribución; en escala logarítmica casi desaparecen. Antes de declarar atípico un valor, conviene preguntarse en qué escala tiene sentido la variable.

### 🔴 Reto 4 · La tarifa del Titanic.

**🎯 Objetivo:** aplicar una transformación logarítmica a una variable real con ceros. **Qué se espera aprender:** cuándo usar `log1p` y cómo cambia la noción de atípico al cambiar de escala.

Con `titanic.csv`, compara la asimetría de `fare` y de `np.log1p(fare)`, cuenta los atípicos por la regla del IQR en cada escala y grafica ambas distribuciones. ¿Por qué se usa `log1p` y no `log`? ¿Los pasajeros con tarifas altas son errores?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 4:**
* El logaritmo comprime la cola derecha y vuelve aditivas las diferencias multiplicativas.
* `log1p` cuando hay ceros; la media de los logaritmos corresponde a la media geométrica.
* Antes de declarar un atípico, pregunta en qué escala tiene sentido la variable.

---
## 5. Ejercicio integrador (20 min).

### 🟢 Ejercicio 7 · Describe la masa corporal en un párrafo.

**🎯 Objetivo:** integrar centro, dispersión, forma y atípicos en una descripción. **Qué se espera aprender:** a redactar la descripción de una variable como se pedirá en la Evaluación 1.

Con los resultados de los ejercicios 2 a 6, redacta un párrafo de 4 a 6 frases que describa `body_mass_g`: centro y dispersión **por especie** (con las medidas adecuadas a la forma), forma de la distribución, por qué la distribución global engaña y qué atípicos hay. Usa números redondeados y unidades.

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* Medidas de centro y dispersión, y cuáles son **robustas** (mediana, IQR, MAD).
* La forma de una distribución: asimetría, curtosis y multimodalidad, y cuatro gráficas para verla.
* Que una distribución global bimodal suele ser la **mezcla de grupos**: hay que describir por especie.
* Tres reglas de valores atípicos y por qué el puntaje *z* clásico sufre **enmascaramiento**.
* La transformación logarítmica para variables con asimetría positiva.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 y, si es posible, un reto 🔴.

**Próxima sesión (jue 22 oct):** agregación, relaciones entre variables y visualización efectiva (`04_agregacion_relaciones_visualizacion.ipynb`). Al terminar se publica la **Evaluación semanal 1**.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>